In [1]:
from langgraph.graph import StateGraph, START , END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from typing import TypedDict

In [2]:
load_dotenv()

True

In [3]:
subgraph_llm = ChatGroq(model='openai/gpt-oss-120b')
parent_llm = ChatGroq(model='openai/gpt-oss-120b')

In [8]:
class parent_state(TypedDict):
    question : str
    answer_eng : str
    answer_urdu : str

In [9]:
def translate_response(state: parent_state):
    """You have given a text , you need to translate it into URDU."""
    message = state['answer_eng']
    response = subgraph_llm.invoke(f'Translate this text into URDU : {message}').content

    return {'answer_urdu': response}

In [10]:
subgraph = StateGraph(parent_state)

subgraph.add_node('translate_response', translate_response)

subgraph.add_edge(START, 'translate_response')
subgraph.add_edge('translate_response', END)

subgraph_workflow = subgraph.compile()

In [11]:
def generate_text(state: parent_state):
    question = state['question']
    response = parent_llm.invoke(f'simple give the answer of it : {question}').content

    return {'answer_eng': response} 

In [12]:
parentgraph = StateGraph(parent_state)

parentgraph.add_node('Generate_text', generate_text)
parentgraph.add_node('translate', subgraph_workflow)

parentgraph.add_edge(START, 'Generate_text')
parentgraph.add_edge('Generate_text', 'translate')
parentgraph.add_edge('translate' , END)

parentgraph_workflow = parentgraph.compile()

In [13]:
parentgraph_workflow.invoke({'question':'who is the current PM of pakistan'})

{'question': 'who is the current PM of pakistan',
 'answer_eng': 'The current Prime Minister of Pakistan is **Shehbaz\u202fSharif**.',
 'answer_urdu': 'موجودہ وزیرِ اعظم پاکستان **شہباز شریف** ہیں۔'}